# Level 1 — Task 2: Data Collection and Cleaning
**Internship:** Codveda Technology — Business Analytics  
**Objective:** Collect data from multiple sources, handle missing values, outliers, and standardize data for analysis.  
**Datasets Used:**
- `churn-bigml-80.csv` — Telecom customer churn training set (~2666 rows)
- `churn-bigml-20.csv` — Telecom customer churn test set (~667 rows)
- `iris.csv` — Classic Iris flower dataset (150 rows)

---

## 0. Setup & Imports
All required libraries. Run `pip install pandas numpy matplotlib seaborn scikit-learn` if needed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# ── Plotting style ──────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded successfully ✓')

---
## 1. Data Collection — Loading from Multiple Sources
We load data from CSV files (simulating collection from Excel/database exports and APIs).  
> **Reusability tip:** Change the `DATA_DIR` path below to point to your own data folder.

In [ ]:
import os

# ── Configure your data path here ───────────────────────────────────────────
DATA_DIR = '../data'   # Change this if running from a different directory

# ── Load datasets ────────────────────────────────────────────────────────────
churn_train = pd.read_csv(os.path.join(DATA_DIR, 'churn-bigml-80.csv'))
churn_test  = pd.read_csv(os.path.join(DATA_DIR, 'churn-bigml-20.csv'))
iris        = pd.read_csv(os.path.join(DATA_DIR, '1__iris.csv'))

# ── Merge churn train + test into one full dataset ───────────────────────────
churn_train['split'] = 'train'
churn_test['split']  = 'test'
churn = pd.concat([churn_train, churn_test], ignore_index=True)

print(f'Churn dataset  : {churn.shape[0]:,} rows × {churn.shape[1]} columns')
print(f'Iris dataset   : {iris.shape[0]:,} rows × {iris.shape[1]} columns')

---
## 2. Initial Data Inspection
Before cleaning anything, we audit the raw data — shape, types, nulls, and duplicates.

In [ ]:
def audit_dataframe(df, name='Dataset'):
    """Reusable function: prints a full audit report for any DataFrame."""
    print(f"{'='*60}")
    print(f" AUDIT REPORT: {name}")
    print(f"{'='*60}")
    print(f"Shape          : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"Duplicates     : {df.duplicated().sum():,}")
    print(f"Total nulls    : {df.isnull().sum().sum():,}")
    print()
    null_summary = df.isnull().sum()
    null_pct     = (null_summary / len(df) * 100).round(2)
    summary = pd.DataFrame({
        'dtype'     : df.dtypes,
        'non_null'  : df.notnull().sum(),
        'null_count': null_summary,
        'null_%'    : null_pct,
        'unique'    : df.nunique()
    })
    print(summary.to_string())
    print()

audit_dataframe(churn, 'Churn (Full)')
audit_dataframe(iris,  'Iris')

In [ ]:
print('── Churn: First 5 rows ──')
display(churn.head())
print('\n── Iris: First 5 rows ──')
display(iris.head())

---
## 3. Structured vs Unstructured Data
Understanding what kind of data we are dealing with is a prerequisite for cleaning.

In [ ]:
def classify_columns(df, name='Dataset'):
    """Reusable: classifies each column as numerical, categorical, or boolean."""
    num_cols  = df.select_dtypes(include=['int64','float64']).columns.tolist()
    cat_cols  = df.select_dtypes(include=['object']).columns.tolist()
    bool_cols = df.select_dtypes(include=['bool']).columns.tolist()
    print(f"── {name} ──")
    print(f"  Numerical   ({len(num_cols)}): {num_cols}")
    print(f"  Categorical ({len(cat_cols)}): {cat_cols}")
    print(f"  Boolean     ({len(bool_cols)}): {bool_cols}")
    print()
    return num_cols, cat_cols, bool_cols

churn_num, churn_cat, churn_bool = classify_columns(churn, 'Churn')
iris_num,  iris_cat,  iris_bool  = classify_columns(iris,  'Iris')

---
## 4. Handling Missing Values
We use different strategies depending on data type:  
- **Numerical**: fill with median (robust to outliers)  
- **Categorical**: fill with mode (most frequent value)  
- **Boolean**: fill with False  

To simulate a realistic scenario, we artificially inject 2% missing values first.

In [ ]:
# ── Inject 2% random nulls to simulate real-world missing data ───────────────
np.random.seed(42)
churn_dirty = churn.copy()

for col in ['Account length', 'Total day minutes', 'Customer service calls', 'State']:
    mask = np.random.rand(len(churn_dirty)) < 0.02
    churn_dirty.loc[mask, col] = np.nan

print('Missing values after injection:')
print(churn_dirty[['Account length','Total day minutes',
                    'Customer service calls','State']].isnull().sum())

In [ ]:
def handle_missing_values(df):
    """
    Reusable: fills missing values by column type.
    Returns a cleaned copy of the DataFrame.
    """
    df_clean = df.copy()
    for col in df_clean.columns:
        null_count = df_clean[col].isnull().sum()
        if null_count == 0:
            continue
        dtype = df_clean[col].dtype
        if dtype in ['float64', 'int64']:
            fill_val = df_clean[col].median()
            strategy = f'median ({fill_val:.2f})'
        elif dtype == 'bool':
            fill_val = False
            strategy = 'False'
        else:
            fill_val = df_clean[col].mode()[0]
            strategy = f'mode ("{fill_val}")'
        df_clean[col].fillna(fill_val, inplace=True)
        print(f'  [{col}] — {null_count} nulls filled with {strategy}')
    return df_clean

print('── Handling Missing Values: Churn ──')
churn_clean = handle_missing_values(churn_dirty)

print('\n── Handling Missing Values: Iris ──')
iris_clean = handle_missing_values(iris)

print(f'\nRemaining nulls — Churn : {churn_clean.isnull().sum().sum()}')
print(f'Remaining nulls — Iris  : {iris_clean.isnull().sum().sum()}')

### Visualising Missing Data Before & After

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

before = churn_dirty.isnull().sum()
before = before[before > 0]
after  = churn_clean.isnull().sum()
after  = after[after > 0] if after.sum() > 0 else pd.Series([0], index=['None'])

before.plot(kind='bar', ax=axes[0], color='#e74c3c', edgecolor='white')
axes[0].set_title('Missing Values — BEFORE Cleaning', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

pd.Series({'No missing values': 0}).plot(kind='bar', ax=axes[1],
                                          color='#2ecc71', edgecolor='white')
axes[1].set_title('Missing Values — AFTER Cleaning', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_ylim(0, before.max() * 1.2)

plt.suptitle('Missing Value Treatment — Churn Dataset', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('missing_values_treatment.png', bbox_inches='tight')
plt.show()
print('Chart saved: missing_values_treatment.png')

---
## 5. Handling Duplicates

In [ ]:
def remove_duplicates(df, name='Dataset'):
    """Reusable: detects and removes duplicate rows."""
    before = len(df)
    df_deduped = df.drop_duplicates()
    after = len(df_deduped)
    removed = before - after
    print(f'{name}: {before:,} rows → {after:,} rows | {removed} duplicates removed')
    return df_deduped

churn_clean = remove_duplicates(churn_clean, 'Churn')
iris_clean  = remove_duplicates(iris_clean,  'Iris')

---
## 6. Outlier Detection & Treatment
We use the **IQR (Interquartile Range)** method — the most robust statistical approach for outlier detection.

> **Formula:** Any value below Q1 − 1.5×IQR or above Q3 + 1.5×IQR is flagged as an outlier.

In [ ]:
def detect_outliers_iqr(df, columns):
    """
    Reusable: detects outliers using the IQR method.
    Returns a summary DataFrame.
    """
    results = []
    for col in columns:
        Q1  = df[col].quantile(0.25)
        Q3  = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        n_out = ((df[col] < lower) | (df[col] > upper)).sum()
        results.append({
            'Column'   : col,
            'Q1'       : round(Q1, 2),
            'Q3'       : round(Q3, 2),
            'IQR'      : round(IQR, 2),
            'Lower Bound': round(lower, 2),
            'Upper Bound': round(upper, 2),
            'Outliers' : n_out,
            'Outlier %': round(n_out / len(df) * 100, 2)
        })
    return pd.DataFrame(results).set_index('Column')

num_cols_churn = ['Account length', 'Total day minutes', 'Total day charge',
                  'Customer service calls', 'Total intl calls']

outlier_report = detect_outliers_iqr(churn_clean, num_cols_churn)
print('── Outlier Detection Report (Churn) ──')
display(outlier_report)

In [ ]:
fig, axes = plt.subplots(1, len(num_cols_churn), figsize=(18, 5))

for ax, col in zip(axes, num_cols_churn):
    ax.boxplot(churn_clean[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='#3498db', color='#2c3e50'),
               medianprops=dict(color='#e74c3c', linewidth=2),
               flierprops=dict(marker='o', color='#e74c3c', markersize=4, alpha=0.5))
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Outlier Detection — Box Plots (Churn Dataset)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outlier_boxplots.png', bbox_inches='tight')
plt.show()
print('Chart saved: outlier_boxplots.png')

In [ ]:
def cap_outliers_iqr(df, columns):
    """
    Reusable: caps outliers at the IQR whisker bounds (Winsorization).
    This preserves row count while reducing extreme value influence.
    """
    df_capped = df.copy()
    for col in columns:
        Q1  = df_capped[col].quantile(0.25)
        Q3  = df_capped[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        before = ((df_capped[col] < lower) | (df_capped[col] > upper)).sum()
        df_capped[col] = df_capped[col].clip(lower=lower, upper=upper)
        print(f'  [{col}] — {before} outliers capped to [{lower:.2f}, {upper:.2f}]')
    return df_capped

print('── Capping Outliers (Churn) ──')
churn_clean = cap_outliers_iqr(churn_clean, num_cols_churn)

---
## 7. Standardization & Normalization
- **StandardScaler** (Z-score): mean=0, std=1 — best for ML models like SVM, PCA
- **MinMaxScaler**: scales to [0,1] — best for neural networks and distance-based models

In [ ]:
def scale_features(df, columns, method='standard'):
    """
    Reusable: scales numerical columns.
    method = 'standard' (Z-score) or 'minmax' (0-1 range)
    Returns scaled DataFrame copy + fitted scaler object.
    """
    df_scaled = df.copy()
    scaler = StandardScaler() if method == 'standard' else MinMaxScaler()
    df_scaled[columns] = scaler.fit_transform(df_scaled[columns])
    print(f'{method.upper()} scaling applied to {len(columns)} columns: {columns}')
    return df_scaled, scaler

# Standard scaling for churn numerical features
churn_scaled, churn_scaler = scale_features(churn_clean, num_cols_churn, method='standard')

# MinMax scaling for iris
iris_num_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
iris_scaled, iris_scaler = scale_features(iris_clean, iris_num_cols, method='minmax')

print('\n── Churn: Stats after StandardScaling ──')
display(churn_scaled[num_cols_churn].describe().round(3))

print('\n── Iris: Stats after MinMaxScaling ──')
display(iris_scaled[iris_num_cols].describe().round(3))

In [ ]:
col = 'Total day minutes'
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, data, title, color in zip(
    axes,
    [churn_clean[col], churn_scaled[col]],
    ['Original', 'After StandardScaler'],
    ['#3498db', '#2ecc71']
):
    ax.hist(data, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{title}\n{col}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')

# MinMax for iris
axes[2].hist(iris_scaled['petal_length'], bins=20, color='#e67e22', edgecolor='white', alpha=0.85)
axes[2].set_title('After MinMaxScaler\nIris: petal_length', fontweight='bold', fontsize=10)
axes[2].set_xlabel('Value')

plt.suptitle('Effect of Scaling on Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('scaling_distributions.png', bbox_inches='tight')
plt.show()
print('Chart saved: scaling_distributions.png')

---
## 8. Encoding Categorical Variables
ML models require numerical input. We encode categorical columns using:
- **Label Encoding** for binary categories (Yes/No, True/False)
- **One-Hot Encoding** for multi-class categories (State)

In [ ]:
def encode_categoricals(df, binary_cols, onehot_cols=None):
    """
    Reusable: encodes categorical columns.
    binary_cols  — label encoded (0/1)
    onehot_cols  — one-hot encoded (new columns per category)
    """
    df_enc = df.copy()
    le = LabelEncoder()

    for col in binary_cols:
        if df_enc[col].dtype == 'bool':
            df_enc[col] = df_enc[col].astype(int)
        else:
            df_enc[col] = le.fit_transform(df_enc[col].astype(str))
        print(f'  Label encoded : {col}')

    if onehot_cols:
        df_enc = pd.get_dummies(df_enc, columns=onehot_cols, drop_first=True)
        print(f'  One-hot encoded: {onehot_cols}')

    return df_enc

print('── Encoding Churn Categoricals ──')
churn_encoded = encode_categoricals(
    churn_clean,
    binary_cols=['International plan', 'Voice mail plan', 'Churn'],
    onehot_cols=['State']
)

print(f'\nChurn shape after encoding: {churn_encoded.shape}')
print('New binary columns sample:')
display(churn_encoded[['International plan','Voice mail plan','Churn']].head(3))

---
## 9. Final Cleaned Datasets — Export

In [ ]:
# Save cleaned versions for use in EDA and ML tasks
churn_clean.to_csv('../data/churn_cleaned.csv', index=False)
iris_clean.to_csv('../data/iris_cleaned.csv', index=False)

print('Cleaned datasets saved:')
print('  → ../data/churn_cleaned.csv')
print('  → ../data/iris_cleaned.csv')

print(f'\nFinal shapes:')
print(f'  Churn : {churn_clean.shape}')
print(f'  Iris  : {iris_clean.shape}')

---
## 10. Cleaning Summary

| Step | Action | Result |
|------|--------|--------|
| 1. Load | Merged train + test churn; loaded iris | 3,334 + 150 rows |
| 2. Audit | Inspected dtypes, nulls, duplicates | Baseline established |
| 3. Missing Values | Median/mode fill by dtype | 0 nulls remaining |
| 4. Duplicates | `drop_duplicates()` | Rows verified unique |
| 5. Outliers | IQR detection + Winsorization cap | Extremes smoothed |
| 6. Scaling | StandardScaler (churn), MinMaxScaler (iris) | Ready for ML |
| 7. Encoding | Label + One-Hot encoding | Categoricals numerical |
| 8. Export | Saved clean CSVs | Ready for EDA & modelling |

> **Business Value:** Clean data reduces model error, prevents skewed KPIs, and ensures reliable decision-making. Dirty data is the #1 cause of inaccurate analytics in real-world business systems.